# Server run control: current_internal

Safe notebook control plane for `current_internal` server execution. Defaults are non-destructive: no FoldX/Rosetta/OpenFold/ESM2 execution, no rsync, no completion markers, and no YAML writes unless explicitly enabled.


In [ ]:
# Parameters
from pathlib import Path
import json
import os
import shutil
import subprocess
import sys
from pprint import pprint

ORIGINAL_CWD = Path.cwd()
USER_PROJECT_ROOT = None  # Explicit override if auto-detection fails.
DATASET = "current_internal"

SERVER_CONFIG = "configs/server_paths.yaml"
SERVER_CONFIG_EXAMPLE = "configs/server_paths.example.yaml"
AUTO_CREATE_SERVER_CONFIG = False
SAVE_SERVER_CONFIG_FROM_NOTEBOOK = False
CONFIRM_WRITE_SERVER_CONFIG = False

# Main server path overrides. None means: use YAML value.
SERVER_PROJECT_ROOT = None
SERVER_DATA_ROOT = None
SERVER_RESULTS_ROOT = None
SERVER_LOGS_ROOT = None
SERVER_REPORTS_ROOT = None

# Tool paths. None means: use YAML value.
FOLDX_BINARY = None
ROSETTA_BIN_DIR = None
OPENFOLD_ENV = None
PYTHON_BIN = None

# GPU settings. None means: use YAML value.
CUDA_VISIBLE_DEVICES = None
MAX_PARALLEL_OPENFOLD_JOBS = None

# Safety flags for future execution cells.
RUN_FOLDX_FULL = False
RUN_FOLDX_SMOKE = False
RUN_ROSETTA = False
RUN_OPENFOLD = False
RUN_ESM2 = False
RUN_RSYNC = False
CREATE_COMPLETION_MARKERS = False


def find_repo_root(start_path):
    start_path = Path(start_path).resolve()
    candidates = [start_path] + list(start_path.parents)
    for candidate in candidates:
        if (candidate / "configs/datasets.yaml").exists():
            return candidate
    for candidate in candidates:
        if (candidate / "configs/run_stages.yaml").exists() and (candidate / ".git").exists():
            return candidate
    for candidate in candidates:
        if (candidate / ".git").exists():
            return candidate
    raise RuntimeError(
        "Could not find repository root. Start Jupyter inside OpenFold3_project or set USER_PROJECT_ROOT explicitly. "
        "Refusing to create or write configs/server_paths.yaml."
    )


PROJECT_ROOT = Path(USER_PROJECT_ROOT).expanduser().resolve() if USER_PROJECT_ROOT else find_repo_root(ORIGINAL_CWD)
os.chdir(PROJECT_ROOT)

print("Original cwd:", ORIGINAL_CWD)
print("Detected project root:", PROJECT_ROOT)
print("Final cwd:", Path.cwd())
print("Dataset:", DATASET)
print("Server config:", PROJECT_ROOT / SERVER_CONFIG)


## 1. Server config and path validation

This section reads `configs/server_paths.yaml` from the detected `PROJECT_ROOT` when present. If it is missing, the notebook uses `configs/server_paths.example.yaml` as a read-only fallback and prints the manual creation command.

Parameter rule: `None` means use the YAML value. A non-`None` value is an explicit notebook override for the effective config and, only with both write safety flags enabled, for saving back to YAML.


In [ ]:
try:
    import yaml
except ImportError as exc:
    raise RuntimeError("PyYAML is required for server config cells") from exc


def project_path(path):
    path = Path(path)
    return path if path.is_absolute() else PROJECT_ROOT / path


def load_yaml(path):
    path = project_path(path)
    if not path.exists():
        return {}
    with path.open("r", encoding="utf-8") as handle:
        data = yaml.safe_load(handle) or {}
    if not isinstance(data, dict):
        raise ValueError(f"YAML root must be a mapping: {path}")
    return data


def save_yaml(path, data):
    path = project_path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as handle:
        yaml.safe_dump(data, handle, sort_keys=False, allow_unicode=True)


def deep_update_dict(base, updates):
    result = dict(base or {})
    for key, value in (updates or {}).items():
        if isinstance(value, dict) and isinstance(result.get(key), dict):
            result[key] = deep_update_dict(result[key], value)
        else:
            result[key] = value
    return result


def print_yaml(data):
    print(yaml.safe_dump(data, sort_keys=False, allow_unicode=True))


def ensure_server_config_exists(example_path, config_path, auto_create=False):
    example_path = project_path(example_path)
    config_path = project_path(config_path)
    if config_path.exists():
        print(f"Using server config: {config_path}")
        return config_path
    if not example_path.exists():
        raise FileNotFoundError(f"Missing example config: {example_path}")
    if auto_create:
        print(f"AUTO_CREATE_SERVER_CONFIG=True; copying {example_path} -> {config_path}")
        config_path.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(example_path, config_path)
        return config_path
    print("configs/server_paths.yaml is missing.")
    print("Create it manually on this machine/server:")
    print(f"cp {example_path} {config_path}")
    print(f"nano {config_path}")
    print("This file is gitignored and should contain local/server-specific paths.")
    return None


def _non_none_overrides():
    updates = {"server": {}, "tools": {}, "gpu": {}}
    mapping = [
        ("server", "project_root", SERVER_PROJECT_ROOT),
        ("server", "data_root", SERVER_DATA_ROOT),
        ("server", "results_root", SERVER_RESULTS_ROOT),
        ("server", "logs_root", SERVER_LOGS_ROOT),
        ("server", "reports_root", SERVER_REPORTS_ROOT),
        ("tools", "foldx_binary", FOLDX_BINARY),
        ("tools", "rosetta_bin_dir", ROSETTA_BIN_DIR),
        ("tools", "openfold_env", OPENFOLD_ENV),
        ("tools", "python_bin", PYTHON_BIN),
        ("gpu", "cuda_visible_devices", CUDA_VISIBLE_DEVICES),
        ("gpu", "max_parallel_openfold_jobs", MAX_PARALLEL_OPENFOLD_JOBS),
    ]
    for section, key, value in mapping:
        if value is not None:
            updates[section][key] = value
    return {section: values for section, values in updates.items() if values}


def load_server_config():
    config_path = project_path(SERVER_CONFIG)
    example_path = project_path(SERVER_CONFIG_EXAMPLE)
    ensure_server_config_exists(example_path, config_path, AUTO_CREATE_SERVER_CONFIG)
    if config_path.exists():
        return load_yaml(config_path), {"path": str(config_path), "exists": True, "example_fallback": False}
    fallback = load_yaml(example_path)
    return fallback, {"path": str(example_path), "requested_path": str(config_path), "exists": False, "example_fallback": True}


def apply_notebook_overrides_to_server_config(config):
    return deep_update_dict(config, _non_none_overrides())


def _path_status(value, kind):
    if value is None or str(value).strip() == "":
        return {"status": "not_set", "is_file": False, "is_dir": False, "writable": False, "executable": False, "notes": ""}
    path = Path(str(value)).expanduser()
    exists = path.exists()
    is_file = path.is_file()
    is_dir = path.is_dir()
    writable = False
    executable = False
    notes = ""
    if kind in {"writable_dir", "dir"} and exists and is_dir:
        writable = os.access(path, os.W_OK)
    if kind == "executable":
        executable = shutil.which(str(value)) is not None if os.sep not in str(value) else os.access(path, os.X_OK)
    status = "exists" if exists or (kind == "executable" and executable) else "missing"
    if kind == "env_name":
        status = "configured" if str(value).strip() else "not_set"
    return {"status": status, "is_file": is_file, "is_dir": is_dir, "writable": writable, "executable": executable, "notes": notes}


def validate_server_paths(config):
    rows = []
    checks = [
        ("server.project_root", config.get("server", {}).get("project_root"), "dir"),
        ("server.data_root", config.get("server", {}).get("data_root"), "dir"),
        ("server.results_root", config.get("server", {}).get("results_root"), "writable_dir"),
        ("server.logs_root", config.get("server", {}).get("logs_root"), "writable_dir"),
        ("server.reports_root", config.get("server", {}).get("reports_root"), "writable_dir"),
        ("tools.foldx_binary", config.get("tools", {}).get("foldx_binary"), "executable"),
        ("tools.rosetta_bin_dir", config.get("tools", {}).get("rosetta_bin_dir"), "dir"),
        ("tools.openfold_env", config.get("tools", {}).get("openfold_env"), "env_name"),
        ("tools.python_bin", config.get("tools", {}).get("python_bin"), "executable"),
        ("gpu.cuda_visible_devices", config.get("gpu", {}).get("cuda_visible_devices"), "value"),
        ("gpu.max_parallel_openfold_jobs", config.get("gpu", {}).get("max_parallel_openfold_jobs"), "value"),
        ("runtime.default_workers_cpu", config.get("runtime", {}).get("default_workers_cpu"), "value"),
        ("runtime.default_workers_gpu", config.get("runtime", {}).get("default_workers_gpu"), "value"),
        ("runtime.use_tmux", config.get("runtime", {}).get("use_tmux"), "value"),
        ("runtime.tmux_session_prefix", config.get("runtime", {}).get("tmux_session_prefix"), "value"),
    ]
    for field, value, kind in checks:
        info = _path_status(value, kind)
        rows.append({"field": field, "value": value, "status": info["status"], "kind": kind, "writable": info["writable"], "executable": info["executable"], "notes": info["notes"]})
    return rows

print("YAML helpers loaded")


In [ ]:
# Safe creation cell. Does not create configs/server_paths.yaml unless AUTO_CREATE_SERVER_CONFIG=True.
if AUTO_CREATE_SERVER_CONFIG:
    ensure_server_config_exists(SERVER_CONFIG_EXAMPLE, SERVER_CONFIG, auto_create=True)
else:
    print("AUTO_CREATE_SERVER_CONFIG is False. To create config manually:")
    print(f"cp {SERVER_CONFIG_EXAMPLE} {SERVER_CONFIG}")


In [ ]:
server_config_raw, server_config_meta = load_server_config()
server_config_effective = apply_notebook_overrides_to_server_config(server_config_raw)
print("Server config metadata:")
pprint(server_config_meta)
print("
Effective server config:")
print_yaml(server_config_effective)


In [ ]:
path_check_rows = validate_server_paths(server_config_effective)
try:
    import pandas as pd
    path_checks_df = pd.DataFrame(path_check_rows)
    display(path_checks_df)
except Exception:
    print("| field | value | status | kind | writable | executable | notes |")
    print("|---|---|---|---|---:|---:|---|")
    for row in path_check_rows:
        print(f"| {row['field']} | `{row['value']}` | {row['status']} | {row['kind']} | {row['writable']} | {row['executable']} | {row['notes']} |")


In [ ]:
# Safe save cell. Requires both SAVE_SERVER_CONFIG_FROM_NOTEBOOK and CONFIRM_WRITE_SERVER_CONFIG.
if SAVE_SERVER_CONFIG_FROM_NOTEBOOK:
    if not CONFIRM_WRITE_SERVER_CONFIG:
        raise RuntimeError("Refusing to write server config: CONFIRM_WRITE_SERVER_CONFIG is False")
    config_path = project_path(SERVER_CONFIG)
    if not config_path.exists() and not AUTO_CREATE_SERVER_CONFIG:
        raise RuntimeError("Refusing to write: configs/server_paths.yaml is missing and AUTO_CREATE_SERVER_CONFIG is False")
    if not config_path.exists():
        ensure_server_config_exists(SERVER_CONFIG_EXAMPLE, SERVER_CONFIG, auto_create=True)
    updates = _non_none_overrides()
    print("Applying non-None notebook overrides:")
    print_yaml(updates)
    new_config = apply_notebook_overrides_to_server_config(load_yaml(config_path))
    save_yaml(config_path, new_config)
    print(f"Saved {config_path}. Reloaded effective YAML:")
    print_yaml(load_yaml(config_path))
else:
    print("SAVE_SERVER_CONFIG_FROM_NOTEBOOK is False; not writing configs/server_paths.yaml")


## 2. CLI dry-runs using SERVER_CONFIG

All commands below pass `--server-config {SERVER_CONFIG}` when the script supports it. These cells are dry-run/control checks only.


In [ ]:
def run_command(command, execute=False):
    print("$", " ".join(map(str, command)))
    if execute:
        return subprocess.run(command, cwd=PROJECT_ROOT, text=True, check=False)
    print("Not executed. Set execute=True in this cell call if you intentionally want to run it.")

COMMAND_PYTHON = server_config_effective.get("tools", {}).get("python_bin") or "python3"
CHECK_ENV_ALL = [COMMAND_PYTHON, "scripts/run/check_server_environment.py", "--stage-group", "all", "--server-config", SERVER_CONFIG]
RUN_PREPARE_DRY = [COMMAND_PYTHON, "scripts/run/run_stage_group.py", "--stage-group", "preprepare", "--dataset", DATASET, "--dry-run", "--server-config", SERVER_CONFIG]
RUN_CPU_DRY = [COMMAND_PYTHON, "scripts/run/run_stage_group.py", "--stage-group", "cpu_heavy", "--dataset", DATASET, "--dry-run", "--server-config", SERVER_CONFIG]
RUN_GPU_DRY = [COMMAND_PYTHON, "scripts/run/run_stage_group.py", "--stage-group", "gpu_heavy", "--dataset", DATASET, "--dry-run", "--server-config", SERVER_CONFIG]
BUNDLE_CPU_DRY = [COMMAND_PYTHON, "scripts/run/build_run_bundle.py", "--dataset", DATASET, "--stage-group", "cpu_heavy", "--dry-run", "--server-config", SERVER_CONFIG]

for cmd in [CHECK_ENV_ALL, RUN_PREPARE_DRY, RUN_CPU_DRY, RUN_GPU_DRY, BUNDLE_CPU_DRY]:
    run_command(cmd, execute=False)
    print()


## 3. FoldX controls

These cells are intentionally disabled by default. Configure `configs/tools/foldx.yaml` and use preflight/smoke-run before any batch run.


In [ ]:
FOLDX_DRY_RUN = [COMMAND_PYTHON, "scripts/executors/run_foldx_experimental.py", "--dataset", DATASET, "--dry-run"]
FOLDX_PREFLIGHT = [COMMAND_PYTHON, "scripts/executors/run_foldx_experimental.py", "--dataset", DATASET, "--preflight"]
FOLDX_SMOKE = [COMMAND_PYTHON, "scripts/executors/run_foldx_experimental.py", "--dataset", DATASET, "--smoke-run", "--max-jobs", "1"]
FOLDX_FULL = [COMMAND_PYTHON, "scripts/executors/run_foldx_experimental.py", "--dataset", DATASET, "--run", "--workers", str(server_config_effective.get("runtime", {}).get("default_workers_cpu", 4)), "--resume"]

run_command(FOLDX_DRY_RUN, execute=False)
run_command(FOLDX_PREFLIGHT, execute=False)
if RUN_FOLDX_SMOKE:
    run_command(FOLDX_SMOKE, execute=True)
else:
    print("RUN_FOLDX_SMOKE is False; smoke-run not executed")
if RUN_FOLDX_FULL:
    run_command(FOLDX_FULL, execute=True)
else:
    print("RUN_FOLDX_FULL = False; full FoldX batch not executed")


## 4. GPU/OpenFold controls

OpenFold is disabled by default in this notebook. Use this section only after server path validation and GPU environment checks pass.


In [ ]:
if RUN_OPENFOLD:
    print("RUN_OPENFOLD is True, but concrete OpenFold executor is not implemented in this notebook yet.")
else:
    print("RUN_OPENFOLD = False; OpenFold is not executed")
